In [ ]:
# Imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Baseline Training & Anomaly Thresholding (2.3)

In [ ]:
# Load the dataset
NORMAL_TRAIN_SET = 'path_to_normal_train_set.csv'  # Update with normal train set path
MODEL_DIR = 'models/'  # Update with model directory
MODEL_NAME = 'baseline_model.h5'  # Update with model name
THRESHOLD_PERCENTILE = 95  # Update with  desired threshold percentile

In [ ]:
# Define baseline model architecture
def create_baseline_model(input_dim):
    # Replace this with Nish's actual model architecture
    model = keras.Sequential([
        keras.layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(input_dim)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

**Tempearyly** load data

In [ ]:


sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Keep all raw-data dependencies together so the notebook is easy to retarget.
DATA_PATH = Path("../data/raw/kddcup.data_10_percent_corrected")
NAMES_PATH = Path("../data/raw/kddcup.names")
ATTACK_TYPES_PATH = Path("../data/raw/training_attack_types")

# Fall back to the original 10 percent export if the corrected file is unavailable.
if not DATA_PATH.exists():
    fallback_path = Path("../data/raw/kddcup.data_10_percent.gz")
    if fallback_path.exists():
        DATA_PATH = fallback_path

# Stop early if a required source file is missing.
for required_path in [DATA_PATH, NAMES_PATH, ATTACK_TYPES_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file not found: {required_path}")

print(f"Dataset path: {DATA_PATH}")
print(f"Column metadata path: {NAMES_PATH}")
print(f"Attack taxonomy path: {ATTACK_TYPES_PATH}")


Dataset path: ../data/raw/kddcup.data_10_percent_corrected
Column metadata path: ../data/raw/kddcup.names
Attack taxonomy path: ../data/raw/training_attack_types


In [ ]:
def load_kdd_data_dictionary(names_path: Path) -> pd.DataFrame:
    """Parse the canonical KDD99 feature definitions from kddcup.names."""
    raw_lines = names_path.read_text().splitlines()
    feature_rows = []

    # The first line lists the label names; the feature definitions start after that.
    for line in raw_lines[1:]:
        cleaned_line = line.strip()
        if not cleaned_line or ":" not in cleaned_line:
            continue
        feature_name, feature_type = cleaned_line.split(":", maxsplit=1)
        feature_rows.append({
            "feature_name": feature_name.strip(),
            "declared_type": feature_type.strip().rstrip("."),
        })

    return pd.DataFrame(feature_rows)


feature_dictionary = load_kdd_data_dictionary(NAMES_PATH)
column_names = feature_dictionary["feature_name"].tolist() + ["label"]
categorical_features = feature_dictionary.loc[
    feature_dictionary["declared_type"].eq("symbolic"), "feature_name"
].tolist()

# This assertion guards against silent schema drift when the raw file changes.
expected_column_count = 42
if len(column_names) != expected_column_count:
    raise ValueError(
        f"Expected {expected_column_count} columns from the metadata files, found {len(column_names)}"
    )

df = pd.read_csv(DATA_PATH, header=None, names=column_names)


Loaded 494,021 rows and 42 columns
Categorical features from the data dictionary: ['protocol_type', 'service', 'flag', 'land', 'logged_in', 'is_host_login', 'is_guest_login']


,feature_name,declared_type
0,duration,continuous
1,protocol_type,symbolic
2,service,symbolic
3,flag,symbolic
4,src_bytes,continuous


In [ ]:
# (Temporary) Create training dataset from model 
# Load normal training data
normal_train_set = df.loc[df["label"] == "normal.", df.columns != "label"].copy()

# Convert categorical features to numeric (one-hot encoding)
categorical_features = feature_dictionary.loc[feature_dictionary["declared_type"].eq("symbolic"), "feature_name"].tolist()
normal_train_set = pd.get_dummies(normal_train_set, columns=categorical_features)

# Scale numeric features
scaler = StandardScaler()
numeric_features = normal_train_set.select_dtypes(include=["number"]).columns
normal_train_set[numeric_features] = scaler.fit_transform(normal_train_set[numeric_features])

# Output the normal train set
print(f"Loaded {len(normal_train_set):,} rows and {len(normal_train_set.columns)} columns")
print("Normal train set shape:", normal_train_set.shape)
print("Normal train set columns:", normal_train_set.columns)

# Split data into training and validation sets
train_set, val_set = train_test_split(normal_train_set_scaled, test_size=0.2, shuffle=False)


In [ ]:
# Create and train baseline model
input_dim = train_set.shape[1]
baseline_model = create_baseline_model(input_dim)
baseline_model.fit(train_set, train_set, epochs=100, batch_size=32, validation_data=(val_set, val_set))

## Anomoly Thresholding

In [ ]:
# Calculate reconstruction errors
reconstruction_errors = np.mean((baseline_model.predict(train_set) - train_set) ** 2, axis=1) # Replace with fucntion

# Calculate anomaly threshold
threshold = np.percentile(reconstruction_errors, THRESHOLD_PERCENTILE)
print(f'Anomaly threshold: {threshold}')

# Plot reconstruction errors histogram
plt.hist(reconstruction_errors, bins=50)
plt.xlabel('Reconstruction Error')
plt.ylabel('Frequency')
plt.title('Reconstruction Errors Histogram')
plt.show()